# 02: Ozaki Schemes

High-precision matmul from low-precision mul blocks.

- **Ozaki I:** split A and B into $k$ slices each, compute $k^2$ cross-product matmuls, acc in FP64
- **Ozaki II:** CRT-based. L modular integer matmuls + reconstruction. Linear cost, not quadratic.

In [1]:
import sys
import pathlib

sys.path.insert(0, str(pathlib.Path("../src")))

In [ ]:
import torch
import math
from fp_emulation.ozaki import (
    extract_slice,
    ozaki_split,
    ozaki1_matmul,
    ozaki2_matmul,
    row_max_abs,
)
from reference import compensated_matmul

## Splitting

Veltkamp works on scalars. Ozaki extends it to matrices:

1. For each row, find max abs value $\mu$
2. Pick $\sigma = 2^{\lceil \log_2(\mu) \rceil + 27}$ (power of two, $\sim 2^{27}$ bigger than biggest element)
3. `slice = (sigma + A) - sigma` rounds every element to $\sim 26$ significant bits
4. `remainder = A - slice`
5. Repeat on remainder

After $k$ rounds: $A = S_1 + S_2 + \ldots + S_k$, each slice has bounded-bit elements.

## extract_slice

Single row, one extraction round, sigma forces rounding.

In [3]:
row = torch.tensor([[3.14159, 0.00272, 100.0, -42.13]], dtype=torch.float64)
mu = row_max_abs(row)

print(f"row: {row[0].tolist()}")
print(f"mu:  {mu[0].item()}")
print()

S, R = extract_slice(row, mu)

print(f"{'element':>10s} {'slice':>12s} {'remainder':>12s}")
for i in range(row.shape[1]):
    print(f"{row[0, i]:>10.5f} {S[0, i]:>12.5f} {R[0, i]:>12.2e}")

print()
print("small elements (0.00272) coarsely rounded relative to mu")
print("to be captured better in the next slice")

row: [3.14159, 0.00272, 100.0, -42.13]
mu:  100.0

   element        slice    remainder
   3.14159      3.14159    -1.18e-07
   0.00272      0.00272     1.21e-07
 100.00000    100.00000     0.00e+00
 -42.13000    -42.13000    -8.39e-07

small elements (0.00272) coarsely rounded relative to mu
to be captured better in the next slice


## Slicing

Remainder shrinks with each slice. More slices = more precision captured.

In [4]:
A = torch.tensor([[3.14159265, 0.00271828, 100.0, -42.1337]], dtype=torch.float64)
n_slices = 4

slices = ozaki_split(A, n_slices)

print(f"{'':>10s}", end="")
for j in range(A.shape[1]):
    print(f" {'col ' + str(j):>12s}", end="")

print()
for i, s in enumerate(slices):
    label = f"slice {i}" if i < n_slices - 1 else "remainder"
    print(f"{label:>10s}", end="")
    for j in range(A.shape[1]):
        print(f" {s[0, j]:>12.4e}", end="")

    print()

print()
reconstruction = sum(slices)
print(f"sum of slices == A: {torch.allclose(reconstruction, A)}")

                  col 0        col 1        col 2        col 3
   slice 0   3.1416e+00   2.7199e-03   1.0000e+02  -4.2134e+01
   slice 1  -1.2831e-06  -1.5992e-06   0.0000e+00  -5.8289e-07
   slice 2  -6.6613e-15  -1.0509e-14   0.0000e+00   7.1054e-15
 remainder   0.0000e+00   0.0000e+00   0.0000e+00   0.0000e+00

sum of slices == A: True


## Ozaki I matmul

`A @ B = sum of all A_slice[i] @ B_slice[j]`

In [5]:
rng = torch.Generator().manual_seed(42)

sizes = [8, 32, 64]
n_slices_list = [2, 3, 4, 6]

for n in sizes:
    A = torch.randn(n, n, dtype=torch.float64, generator=rng)
    B = torch.randn(n, n, dtype=torch.float64, generator=rng)

    C_naive = A @ B
    C_ref = compensated_matmul(A, B)

    err_naive = torch.max(torch.abs(C_naive - C_ref)).item()
    print(f"n={n}:")
    print(f"  {'naive fp64':<20s} max|err| = {err_naive:.3e}")

    for ns in n_slices_list:
        C_oz = ozaki1_matmul(A, B, n_slices=ns)
        err = torch.max(torch.abs(C_oz - C_ref)).item()
        print(
            f"  {'ozaki1 k=' + str(ns):<20s} max|err| = {err:.3e}  ({ns * ns} matmuls)"
        )

    print()

n=8:
  naive fp64           max|err| = 8.882e-16
  ozaki1 k=2           max|err| = 8.882e-16  (4 matmuls)
  ozaki1 k=3           max|err| = 8.882e-16  (9 matmuls)
  ozaki1 k=4           max|err| = 8.882e-16  (16 matmuls)
  ozaki1 k=6           max|err| = 8.882e-16  (36 matmuls)

n=32:
  naive fp64           max|err| = 7.105e-15
  ozaki1 k=2           max|err| = 3.553e-15  (4 matmuls)
  ozaki1 k=3           max|err| = 3.553e-15  (9 matmuls)
  ozaki1 k=4           max|err| = 3.553e-15  (16 matmuls)
  ozaki1 k=6           max|err| = 3.553e-15  (36 matmuls)

n=64:
  naive fp64           max|err| = 1.421e-14
  ozaki1 k=2           max|err| = 7.105e-15  (4 matmuls)
  ozaki1 k=3           max|err| = 7.105e-15  (9 matmuls)
  ozaki1 k=4           max|err| = 7.105e-15  (16 matmuls)
  ozaki1 k=6           max|err| = 7.105e-15  (36 matmuls)



## The tradeoff

| slices ($k$) | matmuls | what happens |
|---|---|---|
| 1 | 1 | same as naive |
| 2 | 4 | captures $\sim 52$ bits, slight improvement |
| 3 | 9 | captures $\sim 78$ bits, plateaus at FP64 acc limit |
| 4+ | 16+ | diminishing returns unless acc is also compensated |

**Bit-width and hardware:** $W$-bit multiplier needs slices with $\leq W/2$ significant bits so products are exact. 26-bit Veltkamp slices need a 52-bit acc (FP64). 8-bit slices need 16-bit ($Q8.8$).

## Ozaki II: Chinese Remainder Theorem

Ozaki I pays $k^2$ matmuls. Ozaki II avoids the cross-product:

1. Scale $A$, $B$ to integers (multiply out the significand, per-row/per-col)
2. Pick $L$ small primes $m_1, m_2, \ldots$ ($\sim 26$-bit each)
3. $C_i = (A \bmod m_i)(B \bmod m_i) \bmod m_i$ for each prime
4. Reconstruct exact integer result via CRT
5. Scale back to float

$L$ matmuls total. CRT recovers $x$ from its residues when $|x| < \prod m_i / 2$. Dot product of 52-bit integers over $n$ elements needs $\sim 104 + \log_2(n)$ bits, so $\sim 5$ primes of 26 bits for $n$ up to $\sim 1000$.

In [ ]:
from fp_emulation.ozaki import _crt, _primes_above

# CRT demo
x = 123456789
moduli = _primes_above(2**26, 5)
residues = [x % m for m in moduli]
recovered = _crt(residues, moduli)

print(f"x = {x}, recovered = {recovered}, match: {x == recovered}")
print(f"moduli cover ~2^{math.log2(math.prod(moduli)):.0f}")

In [7]:
rng = torch.Generator().manual_seed(42)

for n in [8, 16]:
    A = torch.randn(n, n, dtype=torch.float64, generator=rng)
    B = torch.randn(n, n, dtype=torch.float64, generator=rng)

    C_ref = compensated_matmul(A, B)
    C_naive = A @ B
    C_oz1 = ozaki1_matmul(A, B, n_slices=4)
    C_oz2 = ozaki2_matmul(A, B, n_moduli=5)

    err_naive = torch.max(torch.abs(C_naive - C_ref)).item()
    err_oz1 = torch.max(torch.abs(C_oz1 - C_ref)).item()
    err_oz2 = torch.max(torch.abs(C_oz2 - C_ref)).item()

    print(f"n={n}:")
    print(f"  naive:          {err_naive:.3e}")
    print(f"  ozaki I  (16mm): {err_oz1:.3e}")
    print(f"  ozaki II ( 5mm): {err_oz2:.3e}")
    print()

n=8:
  naive:          8.882e-16
  ozaki I  (16mm): 8.882e-16
  ozaki II ( 5mm): 1.776e-15

n=16:
  naive:          5.329e-15
  ozaki I  (16mm): 3.553e-15
  ozaki II ( 5mm): 3.553e-15

